<a href="https://colab.research.google.com/github/GhaithAjaj/GhaithAjaj/blob/main/Filialen_umbuchungen_colab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# ===============================================================
# 1) ERSTEN EXCEL-BERICHT AUS DELTAMASTER TOPM EINLESEN
#    Bericht: "DB1 Rohertrag mit FallPauschalen für Umbuchung (Gesamt)"
#    - Der Nutzer lädt die Datei manuell hoch.
#    - Es wird immer das letzte Tabellenblatt (sheet_name = -1) verwendet.
#    - Die ersten 6 Zeilen enthalten Kopfzeilen / Meta-Infos und werden übersprungen (header=6).
# ===============================================================
print("🔹 Bitte laden Sie den Bericht hoch aus DeltaMaster TopM:")
print("👉 Berichtname: DB1 Rohertrag mit FallPauschalen für Umbuchung (Gesamt)")
uploaded = files.upload()
filename = next(iter(uploaded))
df = pd.read_excel(filename, sheet_name=-1, header=6)

# ===============================================================
# 2) AUFBEREITUNG DES ERSTEN BERICHTES
#    - Die ersten beiden Spalten werden auf einheitliche Bezeichnungen
#      'Hilfsmittel' und 'Filiale' umbenannt.
#    - Aus der Filiale wird die Kostenstelle (KSt) als 5-stelliger Präfix extrahiert.
# ===============================================================
df.rename(columns={df.columns[0]: "Hilfsmittel", df.columns[1]: "Filiale"}, inplace=True)
df["KSt"] = df["Filiale"].astype(str).str[:5]

# ===============================================================
# 3) FILTERUNG DER HILFSMITTEL
#    - Bestimmte Sammelpositionen sollen aus den Berechnungen entfernt werden.
#      Aktuell werden ausgeschlossen:
#      • "Alle Hilfsmittel"
#      • "08 - Einlagen"
#    - Das gefilterte Ergebnis wird in df_clean gespeichert.
# ===============================================================
df_clean = df[~df["Hilfsmittel"].isin(["Alle Hilfsmittel", "08 - Einlagen"])].copy()

# ===============================================================
# 4) BERECHNUNG DER DB-I-PROZENTSÄTZE AUF EINZELPOSITIONSEBENE
#    - (7) DB I % = (6) / (1)
#    - AP DB I % mit FP = AP DB I mit FP / (1) Umsatzberechnung
#    Diese Werte werden zunächst je Hilfsmittel/Filiale berechnet.
# ===============================================================
df_clean["(7) DB I % =\n(6) / (1)"] = df_clean["(6) DB I =\n(1) - (5)"] / df_clean["(1) Umsatz-\nberechnung"]
df_clean["AP DB I % mit FP"] = df_clean["AP DB I mit FP"] / df_clean["(1) Umsatz-\nberechnung"]

# ===============================================================
# 5) MODIFIKATIONEN DER AUFWENDUNGEN (FALLPAUSCHALEN-LOGIK)
#
#    Ziel:
#    - Wir ersetzen die "klassische" DB-I-Logik für bestimmte Hilfsmittel
#      durch eine Variante, in der die Fallpauschalen (AP DB I mit FP)
#      verwendet werden.
#
#    Aktuelle Logik:
#    - Für Hilfsmittel 10 und 18:
#         • "Modifikationen" = "AP DB I mit FP"
#    - Für alle anderen Hilfsmittel:
#         • "Modifikationen" = "(6) DB I = (1) - (5)"
#
#    Parallel dazu wird der prozentuale DB-I-Wert "DB I % Modifikationen"
#    als Modifikationen / (1) Umsatzberechnung berechnet.
# ===============================================================
df_clean["Modifikationen"] = np.where(
    df_clean["Hilfsmittel"].isin(["10 - Gehhilfen", "18 - Kranken-/ Behindertenfahrzeuge"]),
    df_clean["AP DB I mit FP"],
    df_clean["(6) DB I =\n(1) - (5)"]
)
df_clean["DB I % Modifikationen"] = df_clean["Modifikationen"] / df_clean["(1) Umsatz-\nberechnung"]

# ===============================================================
# 5a) ZUSÄTZLICHE VARIANTE: "MODIFIKATIONEN 09 & 32"
#
#    Hintergrund:
#    - Es wird eine zweite Modifikationslogik benötigt, die speziell
#      für die Hilfsmittel
#         • "09 - Elektrostimulationsgeräte"
#         • "32 - Therapeutische Bewegungsgeräte"
#      gilt.
#
#    Definition:
#    - Für alle Hilfsmittel außer 09 und 32:
#         • "Modifikationen 09 & 32" = "Modifikationen"
#    - Für Hilfsmittel 09 und 32:
#         • "Modifikationen 09 & 32" = (1) Umsatzberechnung * 0,80
#
#    Zusätzlich:
#    - "DB I % Modifikationen 09 & 32" = "Modifikationen 09 & 32" / (1) Umsatzberechnung
# ===============================================================
# Startwert: identisch zu der ursprünglichen Spalte "Modifikationen"
df_clean["Modifikationen 09 & 32"] = df_clean["Modifikationen"]

# Maske für Hilfsmittel 09 und 32
mask_0932 = df_clean["Hilfsmittel"].isin([
    "09 - Elektrostimulationsgeräte",
    "32 - Therapeutische Bewegungsgeräte"
])

# Für 09 und 32: 80 % des Umsatzes ansetzen
df_clean.loc[mask_0932, "Modifikationen 09 & 32"] = (
    df_clean.loc[mask_0932, "(1) Umsatz-\nberechnung"] * 0.80
)

# Prozentualer DB-I-Wert auf Basis der neuen Modifikationen-Spalte
df_clean["DB I % Modifikationen 09 & 32"] = (
    df_clean["Modifikationen 09 & 32"] / df_clean["(1) Umsatz-\nberechnung"]
)

# ===============================================================
# 6) AGGREGATION AUF KOSTENSTELLENEBENE (FILIAL-EBENE)
#
#    - Zunächst werden alle numerischen Spalten identifiziert.
#    - Prozent-Spalten (DB-I-Quoten) werden NICHT aufsummiert, sondern
#      später aus den aggregierten Summen neu berechnet.
#    - Alle übrigen numerischen Spalten werden summiert.
# ===============================================================
pct_cols = [
    "(7) DB I % =\n(6) / (1)",
    "AP DB I % mit FP",
    "DB I % Modifikationen",
    "DB I % Modifikationen 09 & 32"
]
sum_cols = [c for c in df_clean.select_dtypes(include="number").columns if c not in pct_cols]

df_filiale = df_clean.groupby("KSt", as_index=False)[sum_cols].sum()

# ===============================================================
# 7) NEUBERECHNUNG DER PROZENTUALEN DB-I-KENNZAHLEN AUF FILIAL-EBENE
#
#    - (7) DB I % = (6) / (1)
#    - AP DB I % mit FP = AP DB I mit FP / (1)
#    - DB I % Modifikationen = Modifikationen / (1)
#    - DB I % Modifikationen 09 & 32 = Modifikationen 09 & 32 / (1)
#
#    Hinweis:
#    - Hier wird die Prozentlogik korrekt aus den aggregierten Summen
#      gebildet und nicht durch Summieren der Prozentwerte.
# ===============================================================
df_filiale["(7) DB I % =\n(6) / (1)"] = df_filiale["(6) DB I =\n(1) - (5)"] / df_filiale["(1) Umsatz-\nberechnung"]
df_filiale["AP DB I % mit FP"] = df_filiale["AP DB I mit FP"] / df_filiale["(1) Umsatz-\nberechnung"]
df_filiale["DB I % Modifikationen"] = df_filiale["Modifikationen"] / df_filiale["(1) Umsatz-\nberechnung"]
df_filiale["DB I % Modifikationen 09 & 32"] = df_filiale["Modifikationen 09 & 32"] / df_filiale["(1) Umsatz-\nberechnung"]

# ===============================================================
# 8) ZUORDNUNG EINER REPRÄSENTATIVEN FILIALE JE KOSTENSTELLE
#
#    - Pro Kostenstelle (KSt) kann es mehrere Filial-Bezeichnungen geben.
#    - Hier wird je KSt der am häufigsten vorkommende Filialname (Modus)
#      ermittelt und als repräsentative "Filiale" in df_filiale ergänzt.
# ===============================================================
filiale_map = df_clean.groupby("KSt")["Filiale"].agg(lambda x: x.mode().iloc[0])
df_filiale["Filiale"] = df_filiale["KSt"].map(filiale_map)

# ===============================================================
# 9) ZWEITEN EXCEL-BERICHT AUS DELTAMASTER ADDISON EINLESEN
#    Bericht: "Rendite Seeger SGE für Umbuchung (Gesamt)"
#
#    - Wieder lädt der Nutzer den Bericht manuell hoch.
#    - Es wird das letzte Tabellenblatt verwendet (sheet_name = -1).
#    - skiprows=8: Die ersten 8 Zeilen sind Kopf-/Meta-Daten.
#    - header=None: Spaltenüberschriften werden manuell gesetzt.
# ===============================================================
print("\n🔹 Bitte laden Sie den Bericht hoch aus DeltaMaster Addison:")
print("👉 Berichtname: Rendite Seeger SGE für Umbuchung (Gesamt)")
uploaded2 = files.upload()
filename2 = next(iter(uploaded2))
df2 = pd.read_excel(filename2, sheet_name=-1, header=None, skiprows=8)

# Spalten sinnvoll benennen und KSt aus Filiale extrahieren
df2.rename(columns={0: "Filiale", 2: "Art", 3: "Wert4", 5: "Wert6"}, inplace=True)
df2["KSt"] = df2["Filiale"].astype(str).str[:5]

# ===============================================================
# 10) FILTERUNG AUF RELEVANTE ARTEN UND PIVOTIERUNG
#
#     - Es werden nur bestimmte Ergebnisarten berücksichtigt:
#       • "Umsatzerlöse"
#       • "Aufwendungen für bez. Lfg. und Lst."
#       • "Rohergebnis"
#     - Für Spalte "Wert4" und "Wert6" werden Pivot-Tabellen je KSt
#       erstellt, um die Werte über alle Zeilen hinweg zu summieren.
#     - Anschließend wird eine kombinierte Tabelle df2_new erzeugt,
#       die sowohl die aktuellen Werte (Wert4) als auch die kumulierten
#       Werte (Wert6) enthält.
# ===============================================================
relevant_arten = ["Umsatzerlöse", "Aufwendungen für bez. Lfg. und Lst.", "Rohergebnis"]
df_sub = df2[df2["Art"].isin(relevant_arten)]

pivot4 = df_sub.pivot_table(index="KSt", columns="Art", values="Wert4", aggfunc="sum")
pivot6 = df_sub.pivot_table(index="KSt", columns="Art", values="Wert6", aggfunc="sum")

df2_new = pivot4.copy()
df2_new["Umsatzerlöse Kum"] = pivot6["Umsatzerlöse"]
df2_new["Aufwendungen für bez. Lfg. und Lst. Kum"] = pivot6["Aufwendungen für bez. Lfg. und Lst."]
df2_new.reset_index(inplace=True)

# ===============================================================
# 11) ZUSAMMENFÜHREN DER BEIDEN DATENQUELLEN
#
#     - Die aggregierten TopM-Daten (df_filiale) werden mit den
#       Addison-Daten (df2_new) über die Kostenstelle (KSt) verknüpft.
#     - Left-Join: Alle Kostenstellen aus df_filiale bleiben erhalten,
#       fehlende Addison-Daten werden als NaN initialisiert.
# ===============================================================
df_merged = df_filiale.merge(df2_new, on="KSt", how="left")

# ===============================================================
# 12) BERECHNUNG "AUFWENDUNGEN FINAL"
#
#     - Fehlende Umsatzerlöse / Aufwendungen werden zunächst mit 0
#       aufgefüllt, um Rechenfehler zu vermeiden.
#     - Formel:
#         Aufwendungen final =
#             Umsatzerlöse * (1 - DB I % Modifikationen)
#             + Aufwendungen für bez. Lfg. und Lst.
#
#       Dabei wird explizit die DB-I-Quote mit Modifikationen (nicht
#       die Variante 09 & 32) verwendet.
#     - Das Ergebnis wird kaufmännisch auf ganze EUR gerundet.
# ===============================================================
df_merged["Umsatzerlöse"] = df_merged["Umsatzerlöse"].fillna(0)
df_merged["Aufwendungen für bez. Lfg. und Lst."] = df_merged["Aufwendungen für bez. Lfg. und Lst."].fillna(0)
df_merged["Aufwendungen final"] = (
    df_merged["Umsatzerlöse"] * (1 - df_merged["DB I % Modifikationen"]) +
    df_merged["Aufwendungen für bez. Lfg. und Lst."]
).round(0)

# ===============================================================
# 13) FORMATIERUNG DER PROZENTWERTE ALS TEXT (z.B. "50,0%")
#
#     - Die DB-I-Quoten werden von Dezimalzahlen (0,5) in formatierte
#       Strings (50,0%) umgewandelt, um die Darstellung in Excel
#       zu vereinheitlichen.
#     - Es werden alle vier prozentualen Kennzahlen formatiert:
#       • (7) DB I % = (6)/(1)
#       • AP DB I % mit FP
#       • DB I % Modifikationen
#       • DB I % Modifikationen 09 & 32
# ===============================================================
for col in [
    "(7) DB I % =\n(6) / (1)",
    "AP DB I % mit FP",
    "DB I % Modifikationen",
    "DB I % Modifikationen 09 & 32"
]:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].apply(
            lambda x: f"{x * 100:.1f}%" if pd.notnull(x) else ""
        )

# ===============================================================
# 14) DEFINIEREN DER SPALTENREIHENFOLGE FÜR DEN EXPORT
#
#     - Hier wird die gewünschte Sortierung der Spalten in der
#       Ergebnisdatei festgelegt.
#     - Zusätzliche Spalten:
#       • "Modifikationen 09 & 32"
#       • "DB I % Modifikationen 09 & 32"
#
#     - Spalten, die in df_merged tatsächlich nicht vorhanden sind,
#       werden später automatisch herausgefiltert.
# ===============================================================
final_cols = [
    "KSt", "Filiale",
    "Aufträge",
    "(1) Umsatz-\nberechnung",
    "(2) Netto EK",
    "(3) Netto EK\nOhne WK",
    "(4) WK EK",
    "AP_EK_Verrechnung_WK_mit_FP",
    "(5) =\n(3) + (4)",
    "(6) DB I =\n(1) - (5)",
    "AP DB I mit FP",
    "Modifikationen",
    "Modifikationen 09 & 32",
    "(7) DB I % =\n(6) / (1)",
    "AP DB I % mit FP",
    "DB I % Modifikationen",
    "DB I % Modifikationen 09 & 32",
    "Umsatzerlöse",
    "Aufwendungen für bez. Lfg. und Lst.",
    "Rohergebnis",
    "Umsatzerlöse Kum",
    "Aufwendungen für bez. Lfg. und Lst. Kum",
    "Aufwendungen final"
]

# Nur Spalten übernehmen, die in df_merged tatsächlich existieren
final_cols_existing = [col for col in final_cols if col in df_merged.columns]
df_export = df_merged[final_cols_existing]

# ===============================================================
# 15) EXPORT NACH EXCEL
#
#     - Die aufbereitete Tabelle wird als "Ergebnis_final_strukturiert.xlsx"
#       mit dem Tabellenblattnamen "Auswertung" gespeichert.
# ===============================================================
output_filename = "Ergebnis_final_strukturiert.xlsx"
df_export.to_excel(output_filename, index=False, sheet_name="Auswertung")

# ===============================================================
# 16) FORMATIERUNG IN EXCEL: GELBE HINTERLEGUNG BESTIMMTER SPALTEN
#
#     - Mit openpyxl wird die erzeugte Excel-Datei erneut geladen.
#     - Die Spalten
#         • "Aufwendungen final"
#         • "DB I % Modifikationen"
#       werden in allen Zeilen gelb hinterlegt, um sie optisch hervorzuheben.
#
#     Hinweis:
#     - Bei Bedarf kann hier die Liste columns_to_color erweitert werden,
#       z.B. um "DB I % Modifikationen 09 & 32".
# ===============================================================
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# Arbeitsmappe und Blatt laden
wb = load_workbook(output_filename)
ws = wb["Auswertung"]

# Gelbe Füllung definieren
yellow_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")

# Kopfzeile auslesen, um Spaltenindizes zu ermitteln
header = [cell.value for cell in ws[1]]
columns_to_color = ["Aufwendungen final", "DB I % Modifikationen"]

# Alle Zeilen in den definierten Spalten gelb einfärben
for col_name in columns_to_color:
    if col_name in header:
        col_idx = header.index(col_name) + 1  # Excel-Spalten sind 1-basiert
        for row in ws.iter_rows(min_row=2, min_col=col_idx, max_col=col_idx):
            for cell in row:
                cell.fill = yellow_fill

# Datei mit Formatierungen speichern
wb.save(output_filename)

# ===============================================================
# 17) DOWNLOAD DER FERTIG FORMATIERTEN EXCEL-DATEI
# ===============================================================
files.download(output_filename)


🔹 Bitte laden Sie den Bericht hoch aus DeltaMaster TopM:
👉 Berichtname: DB1 Rohertrag mit FallPauschalen für Umbuchung (Gesamt)


Saving DB1 Rohertrag mit FallPauschalen colab1.xlsx to DB1 Rohertrag mit FallPauschalen colab1 (1).xlsx

🔹 Bitte laden Sie den Bericht hoch aus DeltaMaster Addison:
👉 Berichtname: Rendite Seeger SGE für Umbuchung (Gesamt)


Saving Rendite Seeger SGE für Umbuchung Colab1.xlsx to Rendite Seeger SGE für Umbuchung Colab1 (1).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>